<a href="https://colab.research.google.com/github/jano87/diplomado-MLunsupervised/blob/main/AllCategoriesFullDiscoveryMLC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MercadoLibre Chile — Which categories move the most volume? (v2)

**Goal:** discover, without knowing them beforehand, which categories in MercadoLibre Chile (`MLC`) move the most sales volume.

## Why v1 returned an empty table
1. **`/sites/MLC/search` is gone for public use** (closed April 2025). It now returns `403 forbidden` even with a valid token. It only works for a seller's own items.
2. **Most of the API now needs an OAuth token.** Calls without one get `403 PolicyAgent`. Changing the User-Agent does not help.
3. **The stock-delta method can't work anymore.** On public resources `available_quantity` is a bucketed reference value and `sold_quantity` was removed, so "units sold = yesterday's stock − today's stock" gives noise.

## What v2 does instead
Every signal comes from an official endpoint that still exists:

| Signal | Endpoint | Token? | What it tells us |
|---|---|---|---|
| Category tree + listing counts | `/categories/{id}` | No | Supply size per category; growth = sellers entering |
| **Best-seller rankings (top 20)** | `/highlights/MLC/category/{id}` | Yes | MercadoLibre's own sales-based ranking |
| Trending searches | `/trends/MLC[/{cat}]` | Yes | Demand: what buyers are looking for |
| Keyword → category | `/sites/MLC/domain_discovery/search` | No | Maps trending searches to categories |
| Reviews (optional) | `/reviews/item/{id}` | Yes | Review growth ≈ sales velocity proxy |

**Key trick (hierarchical attribution):** a parent category's top 20 is a mix of products from its children. If the same product is #1 in *Celulares y Telefonía* and also in the list of *Celulares y Smartphones*, that child "owns" that slot. The share of a parent's best-seller list owned by each child (weighted by rank) is a relative-volume estimate. Then **k-means on PCA-reduced features** groups categories into profiles such as "high-volume movers" or "saturated supply".

**Run this notebook once a day.** Day 1 gives a ranking snapshot. From day 2 on you also get momentum: listing growth, ranking churn and review velocity.

## 0 · One-time setup: MercadoLibre developer credentials
1. Go to https://developers.mercadolibre.cl → *Mis aplicaciones* → **Crear aplicación** (free). Use any HTTPS redirect URI, e.g. `https://www.google.com`.
2. Copy the **App ID** and **Client Secret**.
3. In Colab, open the 🔑 **Secrets** panel (left sidebar) and add `ML_CLIENT_ID` and `ML_CLIENT_SECRET`, with notebook access enabled.
4. *(Fallback)* If the `client_credentials` grant is rejected for your app, do the authorization-code flow once and store the result as `ML_REFRESH_TOKEN`. Instructions are in the auth cell. The notebook saves rotated refresh tokens to the database automatically.

Without credentials the notebook still runs in **public-only mode** (tree + listing counts + supply analysis), but the best-seller and trend signals will be empty.

In [ ]:
# --- CELL 1: SETUP & DATABASE ---
import os, time, json, math, sqlite3, warnings
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import requests
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    BASE_DIR, IN_COLAB = '/content/drive/MyDrive', True
except ImportError:                      # running locally / Jupyter
    BASE_DIR, IN_COLAB = os.environ.get('ML_BASE_DIR', '.'), False

SITE     = 'MLC'
TZ       = ZoneInfo('America/Santiago')
TODAY    = datetime.now(TZ).date().isoformat()   # one snapshot per Chilean calendar day
DB_PATH  = f'{BASE_DIR}/ml_category_discovery_v2.db'

conn = sqlite3.connect(DB_PATH)
conn.executescript("""
CREATE TABLE IF NOT EXISTS categories (
    snapshot_date TEXT, category_id TEXT, name TEXT, parent_id TEXT, root_id TEXT,
    depth INTEGER, total_items INTEGER, n_children INTEGER, path TEXT,
    PRIMARY KEY (snapshot_date, category_id));
CREATE TABLE IF NOT EXISTS highlights (
    snapshot_date TEXT, category_id TEXT, element_id TEXT, element_type TEXT, position INTEGER,
    PRIMARY KEY (snapshot_date, category_id, element_id));
CREATE TABLE IF NOT EXISTS trends (
    snapshot_date TEXT, scope TEXT, keyword TEXT, position INTEGER,
    mapped_category_id TEXT, mapped_l2_id TEXT, mapped_root_id TEXT,
    PRIMARY KEY (snapshot_date, scope, keyword));
CREATE TABLE IF NOT EXISTS reviews (
    snapshot_date TEXT, element_id TEXT, item_id TEXT, total_reviews INTEGER, rating_avg REAL,
    PRIMARY KEY (snapshot_date, element_id));
CREATE TABLE IF NOT EXISTS kv (k TEXT PRIMARY KEY, v TEXT);
""")
conn.commit()
print(f"DB ready: {DB_PATH} | snapshot date: {TODAY}")

In [ ]:
# --- CELL 2: AUTHENTICATED CLIENT (token refresh, retries, rate limiting) ---
def _secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

def kv_get(k):
    row = conn.execute("SELECT v FROM kv WHERE k=?", (k,)).fetchone()
    return row[0] if row else None

def kv_set(k, v):
    conn.execute("INSERT OR REPLACE INTO kv VALUES (?,?)", (k, v)); conn.commit()

class MeliClient:
    API = 'https://api.mercadolibre.com'

    def __init__(self, client_id, client_secret, refresh_token=None, pause=0.15):
        self.client_id, self.client_secret = client_id, client_secret
        self.refresh_token = refresh_token
        self.pause, self.token, self.expires_at = pause, None, 0
        self.calls, self.last_error = 0, None
        self.s = requests.Session()
        self.s.headers.update({'Accept': 'application/json'})

    def authenticate(self):
        if not (self.client_id and self.client_secret):
            print("⚠️  No ML_CLIENT_ID / ML_CLIENT_SECRET → public-only mode.")
            return False
        grants = [{'grant_type': 'client_credentials'}]
        if self.refresh_token:
            grants.append({'grant_type': 'refresh_token', 'refresh_token': self.refresh_token})
        for g in grants:
            r = self.s.post(f'{self.API}/oauth/token',
                            data={**g, 'client_id': self.client_id, 'client_secret': self.client_secret},
                            headers={'content-type': 'application/x-www-form-urlencoded'}, timeout=30)
            if r.ok:
                d = r.json()
                self.token = d['access_token']
                self.expires_at = time.time() + int(d.get('expires_in', 21600)) - 300
                if d.get('refresh_token'):                 # refresh tokens rotate: persist the new one
                    self.refresh_token = d['refresh_token']; kv_set('refresh_token', self.refresh_token)
                print(f"✅ Authenticated via {g['grant_type']} (user_id={d.get('user_id')}, "
                      f"expires in {int(d.get('expires_in', 0))//3600}h)")
                return True
            print(f"   {g['grant_type']} rejected: {r.status_code} {r.text[:160]}")
        return False

    @property
    def has_token(self):
        return self.token is not None

    def get(self, path, params=None, auth=True, retries=4):
        if auth and self.token and time.time() > self.expires_at:
            self.authenticate()
        for attempt in range(retries):
            headers = {'Authorization': f'Bearer {self.token}'} if (auth and self.token) else {}
            try:
                r = self.s.get(self.API + path, params=params, headers=headers, timeout=30)
            except requests.RequestException as e:
                self.last_error = ('network', str(e)[:200]); time.sleep(2 ** attempt); continue
            self.calls += 1
            time.sleep(self.pause)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 401 and auth and attempt == 0 and self.client_id:
                self.authenticate(); continue
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(1.5 * 2 ** attempt); continue
            self.last_error = (r.status_code, r.text[:200])
            return None
        return None

ml = MeliClient(_secret('ML_CLIENT_ID'), _secret('ML_CLIENT_SECRET'),
                refresh_token=kv_get('refresh_token') or _secret('ML_REFRESH_TOKEN'))
ml.authenticate()

# ---- Fallback: authorization-code flow (only if client_credentials is rejected) ----
# 1) Open in your browser (logged in to MercadoLibre):
#    https://auth.mercadolibre.cl/authorization?response_type=code&client_id=<APP_ID>&redirect_uri=<YOUR_REDIRECT_URI>
# 2) Copy the ?code=TG-... from the redirected URL and run once:
#    r = requests.post('https://api.mercadolibre.com/oauth/token', data={'grant_type':'authorization_code',
#        'client_id': ml.client_id, 'client_secret': ml.client_secret, 'code': 'TG-...', 'redirect_uri': '<YOUR_REDIRECT_URI>'})
#    kv_set('refresh_token', r.json()['refresh_token']); ml.refresh_token = kv_get('refresh_token'); ml.authenticate()

In [ ]:
# --- CELL 3: PROBE WHICH SIGNALS ARE AVAILABLE TODAY ---
# MercadoLibre tightens policies per endpoint, so check before collecting.
PROBES = {
    'category_tree':    dict(path='/categories/MLC1051', auth=False),
    'root_list':        dict(path=f'/sites/{SITE}/categories'),
    'highlights':       dict(path=f'/highlights/{SITE}/category/MLC1051'),
    'trends':           dict(path=f'/trends/{SITE}'),
    'domain_discovery': dict(path=f'/sites/{SITE}/domain_discovery/search', params={'q': 'celular', 'limit': 1}, auth=False),
}
CAN = {}
for name, p in PROBES.items():
    res = ml.get(p['path'], params=p.get('params'), auth=p.get('auth', True))
    CAN[name] = res is not None
    print(f"{'✅' if CAN[name] else '❌'} {name:<17} {'' if CAN[name] else ml.last_error}")

if not CAN['highlights']:
    print("\n⚠️  Best-seller rankings unavailable → volume estimates will rely on trends/listings only.")

## 1 · Discover the category tree (no prior knowledge needed)
Walk the tree from the roots down to `MAX_DEPTH`. Each node records `total_items_in_this_category`, the size of its supply.

In [ ]:
# --- CELL 4: CRAWL CATEGORY TREE ---
MAX_DEPTH = 2          # 1 = roots only, 2 = roots + subcategories (~500 nodes), 3 = much larger

# Verified MLC roots (Oct 2026). Used only if /sites/MLC/categories is blocked.
FALLBACK_ROOTS = {
    'MLC1747': 'Accesorios para Vehículos', 'MLC1512': 'Agro', 'MLC1403': 'Alimentos y Bebidas',
    'MLC1367': 'Antigüedades y Colecciones', 'MLC1368': 'Arte, Librería y Cordonería',
    'MLC1743': 'Autos, Motos y Otros', 'MLC1384': 'Bebés', 'MLC1246': 'Belleza y Cuidado Personal',
    'MLC1039': 'Cámaras y Accesorios', 'MLC1051': 'Celulares y Telefonía', 'MLC1648': 'Computación',
    'MLC1144': 'Consolas y Videojuegos', 'MLC1500': 'Construcción', 'MLC1276': 'Deportes y Fitness',
    'MLC5726': 'Electrodomésticos', 'MLC1000': 'Electrónica, Audio y Video', 'MLC178483': 'Herramientas',
    'MLC1574': 'Hogar y Muebles', 'MLC1499': 'Industrias y Oficinas', 'MLC1459': 'Inmuebles',
    'MLC1182': 'Instrumentos Musicales', 'MLC1132': 'Juegos y Juguetes', 'MLC3025': 'Libros, Revistas y Comics',
    'MLC1071': 'Mascotas', 'MLC1168': 'Música y Películas', 'MLC1953': 'Otras Categorías',
    'MLC3937': 'Relojes y Joyas', 'MLC409431': 'Salud y Equipamiento Médico', 'MLC1540': 'Servicios',
    'MLC435280': 'Souvenirs, Cotillón y Fiestas', 'MLC1430': 'Vestuario y Calzado',
}
# Not sales markets for physical goods volume; excluded from ranking (set to [] to keep them)
EXCLUDE_ROOTS = ['MLC1459', 'MLC1743', 'MLC1540']   # real estate, vehicles, services

def get_category(cid):
    return ml.get(f'/categories/{cid}', auth=False) or (ml.get(f'/categories/{cid}') if ml.has_token else None)

def crawl_tree(roots, max_depth):
    rows, frontier = [], [(r['id'], None, r['id'], 1) for r in roots]
    while frontier:
        cid, parent, root, depth = frontier.pop(0)
        d = get_category(cid)
        if not d:
            continue
        kids = d.get('children_categories', [])
        path = ' > '.join(p['name'].strip() for p in d.get('path_from_root', []))
        rows.append(dict(snapshot_date=TODAY, category_id=cid, name=d['name'].strip(), parent_id=parent,
                         root_id=root, depth=depth, total_items=d.get('total_items_in_this_category'),
                         n_children=len(kids), path=path))
        if depth + 1 < max_depth:
            frontier += [(k['id'], cid, root, depth + 1) for k in kids]
        elif depth + 1 == max_depth:        # deepest level: parent response already has name + count
            for k in kids:
                rows.append(dict(snapshot_date=TODAY, category_id=k['id'], name=k['name'].strip(), parent_id=cid,
                                 root_id=root, depth=depth + 1, total_items=k.get('total_items_in_this_category'),
                                 n_children=None, path=f"{path} > {k['name'].strip()}"))
    return pd.DataFrame(rows)

roots = ml.get(f'/sites/{SITE}/categories') if CAN.get('root_list') else None
if not roots:
    roots = [{'id': k, 'name': v} for k, v in FALLBACK_ROOTS.items()]
roots = [r for r in roots if r['id'] not in EXCLUDE_ROOTS]

t0 = time.time()
tree = crawl_tree(roots, MAX_DEPTH)
tree.to_sql('categories_tmp', conn, if_exists='replace', index=False)
conn.execute("INSERT OR REPLACE INTO categories SELECT * FROM categories_tmp"); conn.execute("DROP TABLE categories_tmp"); conn.commit()
print(f"🌳 {len(tree)} categories ({(tree.depth==1).sum()} roots) in {time.time()-t0:.0f}s, {ml.calls} API calls so far")
display(tree[tree.depth == 1].sort_values('total_items', ascending=False)[['category_id', 'name', 'total_items', 'n_children']].head(10))

## 2 · Collect today's demand signals
* **Best sellers**: top 20 for every root and every level-2 category.
* **Trends**: site-wide top 50 searches plus per-root searches, mapped to categories with `domain_discovery`.

In [ ]:
# --- CELL 5: BEST-SELLER RANKINGS (highlights) ---
def collect_highlights(category_ids):
    n_ok = 0
    for i, cid in enumerate(category_ids, 1):
        d = ml.get(f'/highlights/{SITE}/category/{cid}')
        if d and d.get('content'):
            conn.executemany("INSERT OR REPLACE INTO highlights VALUES (?,?,?,?,?)",
                             [(TODAY, cid, e['id'], e.get('type'), e.get('position')) for e in d['content']])
            n_ok += 1
        if i % 100 == 0:
            conn.commit(); print(f"   {i}/{len(category_ids)} categories…")
    conn.commit()
    return n_ok

already = {r[0] for r in conn.execute("SELECT DISTINCT category_id FROM highlights WHERE snapshot_date=?", (TODAY,))}
todo = [c for c in tree.category_id if c not in already]          # resumable if Colab disconnects
if CAN.get('highlights'):
    t0 = time.time()
    ok = collect_highlights(todo)
    print(f"🏆 Rankings stored for {ok}/{len(todo)} new categories ({len(already)} already done today) in {time.time()-t0:.0f}s")
else:
    print("Skipped (no access to /highlights).")

In [ ]:
# --- CELL 6: TRENDING SEARCHES → CATEGORIES ---
_path_cache = {}
def ancestors(cid):
    """Return (l2_id, root_id) for any category id."""
    if cid not in _path_cache:
        d = get_category(cid)
        p = [x['id'] for x in d.get('path_from_root', [])] if d else []
        _path_cache[cid] = (p[1] if len(p) > 1 else (p[0] if p else None), p[0] if p else None)
    return _path_cache[cid]

def map_keyword(kw):
    res = ml.get(f'/sites/{SITE}/domain_discovery/search', params={'q': kw, 'limit': 1}, auth=False)
    if res:
        cid = res[0]['category_id']
        return (cid, *ancestors(cid))
    return (None, None, None)

def collect_trends(scope, path):
    data = ml.get(path)
    if not data:
        return 0
    rows = []
    for pos, t in enumerate(data, 1):
        kw = t.get('keyword')
        if kw:
            rows.append((TODAY, scope, kw, pos, *map_keyword(kw)))
    conn.executemany("INSERT OR REPLACE INTO trends VALUES (?,?,?,?,?,?,?)", rows); conn.commit()
    return len(rows)

if CAN.get('trends'):
    done = {r[0] for r in conn.execute("SELECT DISTINCT scope FROM trends WHERE snapshot_date=?", (TODAY,))}
    n = 0 if 'SITE' in done else collect_trends('SITE', f'/trends/{SITE}')
    for r in roots:
        if r['id'] not in done:
            n += collect_trends(r['id'], f"/trends/{SITE}/{r['id']}")
    print(f"🔎 {n} trending keywords mapped to categories")
    display(pd.read_sql("""SELECT position, keyword, c.name AS l2_category FROM trends t
                           LEFT JOIN categories c ON c.category_id=t.mapped_l2_id AND c.snapshot_date=t.snapshot_date
                           WHERE t.snapshot_date=? AND scope='SITE' ORDER BY position LIMIT 15""", conn, params=(TODAY,)))
else:
    print("Skipped (no access to /trends).")

In [ ]:
# --- CELL 7 (OPTIONAL): REVIEW COUNTS AS A SALES-VELOCITY PROXY ---
# Reviews accumulate roughly in proportion to units sold. Tracking the daily *increase* in review
# counts for the root-level top 20 gives a cross-category comparable velocity signal.
COLLECT_REVIEWS = True
REVIEW_SCOPE_DEPTH = 1      # only root-level best sellers (~20 × #roots elements, 1–2 calls each)

def item_id_for(element_id, element_type):
    if element_type == 'ITEM':
        return element_id
    if element_type == 'PRODUCT':
        p = ml.get(f'/products/{element_id}')
        return ((p or {}).get('buy_box_winner') or {}).get('item_id')
    return None                       # USER_PRODUCT: no stable public mapping, skipped

def collect_reviews():
    els = pd.read_sql("""SELECT DISTINCT h.element_id, h.element_type FROM highlights h
                         JOIN categories c ON c.category_id=h.category_id AND c.snapshot_date=h.snapshot_date
                         WHERE h.snapshot_date=? AND c.depth<=?
                         AND h.element_id NOT IN (SELECT element_id FROM reviews WHERE snapshot_date=?)""",
                      conn, params=(TODAY, REVIEW_SCOPE_DEPTH, TODAY))
    if els.empty:
        return 0
    # probe once
    first = els.iloc[0]; iid = item_id_for(first.element_id, first.element_type)
    if iid is None or ml.get(f'/reviews/item/{iid}') is None:
        print(f"Reviews endpoint not available: {ml.last_error}"); return 0
    n = 0
    for e in els.itertuples():
        iid = item_id_for(e.element_id, e.element_type)
        r = ml.get(f'/reviews/item/{iid}') if iid else None
        if r is not None:
            conn.execute("INSERT OR REPLACE INTO reviews VALUES (?,?,?,?,?)",
                         (TODAY, e.element_id, iid, (r.get('paging') or {}).get('total'), r.get('rating_average')))
            n += 1
    conn.commit()
    return n

if COLLECT_REVIEWS and CAN.get('highlights'):
    print(f"⭐ Review counts stored for {collect_reviews()} best-selling elements")

## 3 · Turn signals into a volume estimate per category
For each **level-2 category `c`** inside root `r`:

* **`parent_share`** — rank-weighted share (weight = 1/position) of `r`'s top 20 whose products also appear in `c`'s top 20. If `c` holds 6 of the top 10 slots in its root, it moves most of that root's volume.
* **`root_weight`** — how big root `r` is compared with the other roots. Uses review velocity if available, otherwise total reviews, otherwise site-wide trending searches, otherwise listing share.
* **`volume_index = parent_share × root_weight`** — comparable across the whole site, and it sums to ~1.
* Supporting features: listings (log), listing growth, trend hits, how deep `c`'s own ranking reaches into the parent list (`depth_in_parent`), and ranking churn.

In [ ]:
# --- CELL 8: FEATURE ENGINEERING ---
def snapshot_dates(table):
    return [r[0] for r in conn.execute(f"SELECT DISTINCT snapshot_date FROM {table} ORDER BY 1")]

def build_features(day=TODAY, lookback_days=7):
    cats = pd.read_sql("SELECT * FROM categories WHERE snapshot_date=?", conn, params=(day,))
    hl   = pd.read_sql("SELECT * FROM highlights WHERE snapshot_date=?", conn, params=(day,))
    tr   = pd.read_sql("SELECT * FROM trends WHERE snapshot_date=?", conn, params=(day,))
    rv   = pd.read_sql("SELECT * FROM reviews", conn)
    roots_df, l2 = cats[cats.depth == 1].copy(), cats[cats.depth == 2].copy()

    # ---- parent_share & depth_in_parent (hierarchical attribution) ----
    hl['w'] = 1.0 / hl['position']
    root_lists = hl[hl.category_id.isin(roots_df.category_id)].rename(columns={'category_id': 'root_id'})
    root_lists['w'] = root_lists.groupby('root_id')['w'].transform(lambda s: s / s.sum())
    l2_lists = hl.merge(l2[['category_id', 'root_id']], on='category_id')
    m = l2_lists.merge(root_lists[['root_id', 'element_id', 'w', 'position']], on=['root_id', 'element_id'],
                       suffixes=('', '_root'))
    share = m.groupby('category_id').agg(parent_share=('w_root', 'sum'),
                                         depth_in_parent=('element_id', 'count'),
                                         best_root_rank=('position_root', 'min')).reset_index()
    coverage = m.groupby('root_id')['w_root'].sum().rename('attribution_coverage')

    # ---- root_weight: pick the best available cross-root signal ----
    root_signal, source = None, None
    if not rv.empty and not root_lists.empty:
        rv = rv.sort_values('snapshot_date')
        last = rv[rv.snapshot_date == day].set_index('element_id')['total_reviews']
        prev_days = [d for d in rv.snapshot_date.unique() if d < day]
        per_el = None
        if prev_days:
            prev = rv[rv.snapshot_date == prev_days[-1]].set_index('element_id')['total_reviews']
            gap = max((pd.Timestamp(day) - pd.Timestamp(prev_days[-1])).days, 1)
            per_el = ((last - prev).clip(lower=0) / gap).dropna()
            source = 'review velocity (reviews/day)'
        if per_el is None or per_el.sum() == 0:
            per_el, source = last.dropna(), 'total reviews of root top-20'
        rl = root_lists.assign(sig=root_lists.element_id.map(per_el)).dropna(subset=['sig'])
        if len(rl) and rl.sig.sum() > 0:
            root_signal = rl.groupby('root_id')['sig'].sum()
    if root_signal is None and not tr.empty:
        st = tr[(tr.scope == 'SITE') & tr.mapped_root_id.notna()]
        if len(st):
            root_signal = (1.0 / np.sqrt(st.position)).groupby(st.mapped_root_id).sum()
            source = 'site-wide trending searches'
    if root_signal is None:
        root_signal, source = roots_df.set_index('category_id')['total_items'].astype(float), 'listing share (supply only)'
    root_signal = root_signal.reindex(roots_df.category_id).fillna(0)
    root_signal = root_signal + 0.02 * root_signal.max()      # smoothing: unseen roots aren't exactly zero
    root_weight = (root_signal / root_signal.sum()).rename('root_weight')

    # ---- trends per L2 (site + per-root keyword lists) ----
    t_l2 = tr[tr.mapped_l2_id.notna()].assign(tw=lambda d: 1.0 / np.sqrt(d.position))
    trend_hits = t_l2.groupby('mapped_l2_id')['tw'].sum().rename('trend_score')

    # ---- listing growth & ranking churn vs. earliest snapshot within lookback ----
    past_days = [d for d in snapshot_dates('categories') if d < day and
                 (pd.Timestamp(day) - pd.Timestamp(d)).days <= lookback_days]
    growth = pd.Series(dtype=float, name='listing_growth')
    churn = pd.Series(dtype=float, name='rank_churn')
    if past_days:
        base = pd.read_sql("SELECT category_id, total_items FROM categories WHERE snapshot_date=?", conn,
                           params=(past_days[0],)).set_index('category_id')['total_items']
        cur = cats.set_index('category_id')['total_items']
        growth = ((cur - base) / base.replace(0, np.nan)).rename('listing_growth')
        prev_hl = pd.read_sql("SELECT category_id, element_id FROM highlights WHERE snapshot_date=?", conn,
                              params=(past_days[-1],))
        if len(prev_hl):
            ps = prev_hl.groupby('category_id')['element_id'].apply(set)
            cs = hl.groupby('category_id')['element_id'].apply(set)
            churn = pd.Series({c: 1 - len(cs[c] & ps[c]) / max(len(cs[c]), 1) for c in cs.index if c in ps.index},
                              name='rank_churn')

    f = (l2.merge(share, on='category_id', how='left')
           .merge(roots_df[['category_id', 'name']].rename(columns={'category_id': 'root_id', 'name': 'root_name'}), on='root_id')
           .merge(root_weight, left_on='root_id', right_index=True, how='left')
           .merge(trend_hits, left_on='category_id', right_index=True, how='left')
           .merge(growth, left_on='category_id', right_index=True, how='left')
           .merge(churn, left_on='category_id', right_index=True, how='left'))
    f[['parent_share', 'depth_in_parent', 'trend_score']] = f[['parent_share', 'depth_in_parent', 'trend_score']].fillna(0)
    f['volume_index'] = f.parent_share * f.root_weight
    f['log_listings'] = np.log1p(f.total_items.fillna(0))
    f['listing_growth'] = f['listing_growth'].fillna(0)
    f['rank_churn'] = f['rank_churn'].fillna(f['rank_churn'].median() if f['rank_churn'].notna().any() else 0)
    f['has_ranking'] = f.category_id.isin(hl.category_id)
    meta = dict(root_weight_source=source, coverage=coverage, n_days=len(snapshot_dates('categories')))
    return f, meta

feat, META = build_features()
print(f"Features for {len(feat)} level-2 categories | root weight source: {META['root_weight_source']}")
cov = META['coverage']
if len(cov):
    print(f"Attribution coverage (share of root top-20 matched to a child): median {cov.median():.0%}")

In [ ]:
# --- CELL 9: TRANSPARENT VOLUME SCORE & RANKING ---
WEIGHTS = {                      # tweak to taste; percentile-ranked so units don't matter
    'volume_index':    0.50,
    'depth_in_parent': 0.15,
    'trend_score':     0.15,
    'listing_growth':  0.10,
    'log_listings':    0.10,
}
if META['n_days'] < 2:                      # growth is all zeros on day 1 → redistribute its weight
    WEIGHTS['volume_index'] += WEIGHTS.pop('listing_growth')

feat['volume_score'] = sum(feat[k].rank(pct=True) * w for k, w in WEIGHTS.items()) / sum(WEIGHTS.values()) * 100
feat['est_share_of_site_volume'] = feat.volume_index / feat.volume_index.sum() if feat.volume_index.sum() > 0 else 0

top = feat.sort_values(['volume_score', 'volume_index'], ascending=False)
cols = ['root_name', 'name', 'volume_score', 'est_share_of_site_volume', 'parent_share', 'depth_in_parent',
        'best_root_rank', 'trend_score', 'total_items', 'listing_growth']
if not CAN.get('highlights'):
    print("⚠️  PUBLIC-ONLY MODE: no best-seller data, so this ranking reflects supply (listing counts), not sales volume.")
print("🔥 Top 25 level-2 categories by estimated sales volume")
display(top[cols].head(25).style.format({'volume_score': '{:.1f}', 'est_share_of_site_volume': '{:.1%}',
                                         'parent_share': '{:.0%}', 'trend_score': '{:.2f}',
                                         'total_items': '{:,.0f}', 'listing_growth': '{:+.1%}', 'best_root_rank': '{:.0f}'}))

## 4 · Unsupervised profiling: PCA + k-means
Rankings answer "who is biggest". Clustering answers "what kind of market is this?". Typical profiles are:
* **high volume, strong demand** (big `volume_index` and trend score)
* **saturated supply** (huge listing counts, little best-seller presence)
* **emerging** (growing listings or high churn, small current share)
* **long tail**

In [ ]:
# --- CELL 10: PCA + K-MEANS ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

FEATURES = ['volume_index', 'parent_share', 'depth_in_parent', 'trend_score', 'log_listings', 'listing_growth', 'rank_churn']
FEATURES = [c for c in FEATURES if feat[c].std() > 0]          # drop constant features (e.g. growth on day 1)

if len(FEATURES) < 3:
    print("⚠️  Not enough informative features to cluster (public-only mode?). "
          "Add ML_CLIENT_ID / ML_CLIENT_SECRET to enable best-seller and trend signals.")
else:

    X = feat[FEATURES].copy()
    for c in ['volume_index', 'trend_score']:                       # heavy-tailed → log scale
        if c in X: X[c] = np.log1p(X[c] * 100)
    Xs = StandardScaler().fit_transform(X)

    MIN_K, MAX_K = 3, 7          # k=2 usually wins silhouette but just splits 'big' vs 'rest'; 3+ gives usable profiles
    scores = {k: silhouette_score(Xs, KMeans(k, n_init=10, random_state=42).fit_predict(Xs)) for k in range(MIN_K, MAX_K + 1)}
    K = max(scores, key=scores.get)
    km = KMeans(K, n_init=10, random_state=42).fit(Xs)
    feat['cluster'] = km.labels_
    print("Silhouette by k:", {k: round(v, 3) for k, v in scores.items()}, "→ k =", K)

    # name clusters by their profile
    prof = feat.groupby('cluster').agg(n=('name', 'size'), volume_score=('volume_score', 'mean'),
                                       vol_share=('est_share_of_site_volume', 'sum'), listings=('total_items', 'median'),
                                       trend=('trend_score', 'mean'), growth=('listing_growth', 'mean'))
    order = prof.volume_score.sort_values(ascending=False).index
    names = {}
    for i, c in enumerate(order):
        p = prof.loc[c]
        if i == 0:
            names[c] = '🔥 High-volume movers'
        elif p.listings > prof.listings.median() * 1.5 and p.volume_score < prof.volume_score.median():
            names[c] = '📦 Saturated supply'
        elif p.growth > prof.growth.median() and p.growth > 0:
            names[c] = '🌱 Emerging'
        elif i == len(order) - 1:
            names[c] = '🪶 Long tail'
        else:
            names[c] = f'Mid-tier #{i}'
    feat['profile'] = feat.cluster.map(names)
    prof.index = prof.index.map(names)
    display(prof.sort_values('volume_score', ascending=False).style.format(
        {'volume_score': '{:.1f}', 'vol_share': '{:.1%}', 'listings': '{:,.0f}', 'trend': '{:.2f}', 'growth': '{:+.1%}'}))

    pca = PCA(2).fit(Xs); P = pca.transform(Xs)
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    for prof_name, g in feat.assign(pc1=P[:, 0], pc2=P[:, 1]).groupby('profile'):
        ax[0].scatter(g.pc1, g.pc2, s=20 + 400 * g.est_share_of_site_volume / max(feat.est_share_of_site_volume.max(), 1e-9),
                      alpha=.7, label=prof_name)
    for _, r in feat.assign(pc1=P[:, 0], pc2=P[:, 1]).nlargest(10, 'volume_score').iterrows():
        ax[0].annotate(r['name'][:28], (r.pc1, r.pc2), fontsize=8)
    ax[0].set(title=f'Category profiles (PCA, {pca.explained_variance_ratio_.sum():.0%} variance)', xlabel='PC1', ylabel='PC2')
    ax[0].legend(fontsize=8)
    loadings = pd.DataFrame(pca.components_.T, index=FEATURES, columns=['PC1', 'PC2'])
    loadings.plot.barh(ax=ax[1], title='What the PCA axes mean (loadings)')
    plt.tight_layout(); plt.show()

    print("\nTop categories in the high-volume cluster:")
    display(feat[feat.profile == '🔥 High-volume movers'].sort_values('volume_score', ascending=False)[['root_name', 'name', 'volume_score', 'est_share_of_site_volume']].head(15))

## 5 · Drill down into the winners
Apply the same attribution one level deeper to find the exact niches (level 3+) inside the top categories.

In [ ]:
# --- CELL 11: DRILL-DOWN TO NICHES ---
DRILL_TOP_N = 8

def drill_down(parent_id, parent_volume):
    parent = get_category(parent_id)
    plist = ml.get(f'/highlights/{SITE}/category/{parent_id}')
    if not parent or not plist:
        return []
    pw = {e['id']: 1 / e['position'] for e in plist['content']}
    tot = sum(pw.values())
    out = []
    for k in parent.get('children_categories', []):
        cl = ml.get(f'/highlights/{SITE}/category/{k["id"]}')
        ids = [e['id'] for e in (cl or {}).get('content', [])]
        sh = sum(pw.get(i, 0) for i in ids) / tot
        out.append(dict(parent=parent['name'], niche=k['name'], niche_id=k['id'], share_of_parent=sh,
                        est_share_of_site_volume=sh * parent_volume, listings=k.get('total_items_in_this_category'),
                        slots_in_parent_top20=sum(i in pw for i in ids)))
    return out

if CAN.get('highlights'):
    rows = []
    for _, r in top.head(DRILL_TOP_N).iterrows():
        rows += drill_down(r.category_id, r.est_share_of_site_volume)
    niches = pd.DataFrame(rows).sort_values('est_share_of_site_volume', ascending=False)
    print("🎯 Hottest niches (level 3)")
    display(niches.head(25).style.format({'share_of_parent': '{:.0%}', 'est_share_of_site_volume': '{:.2%}', 'listings': '{:,.0f}'}))

## 6 · Momentum over time (needs ≥ 2 daily runs)

In [ ]:
# --- CELL 12: MOMENTUM ACROSS SNAPSHOTS ---
days = [d for d in snapshot_dates('highlights')]
if len(days) < 2:
    print(f"Only {len(days)} snapshot(s) so far. Run this notebook again tomorrow to unlock momentum.")
else:
    hist = []
    for d in days[-14:]:
        f, _ = build_features(d)
        f['est_share'] = f.volume_index / f.volume_index.sum()
        hist.append(f[['category_id', 'name', 'root_name', 'est_share', 'total_items']].assign(day=d))
    hist = pd.concat(hist)
    piv = hist.pivot_table(index=['root_name', 'name'], columns='day', values='est_share')
    first, last = piv.columns[0], piv.columns[-1]
    mom = pd.DataFrame({'share_first': piv[first], 'share_last': piv[last]})
    mom['delta_pp'] = (mom.share_last - mom.share_first) * 100
    print(f"📈 Biggest gainers in estimated volume share ({first} → {last})")
    display(mom.sort_values('delta_pp', ascending=False).head(15).style.format(
        {'share_first': '{:.2%}', 'share_last': '{:.2%}', 'delta_pp': '{:+.2f} pp'}))
    top10 = mom.share_last.nlargest(10).index
    piv.loc[top10].T.plot(figsize=(12, 5), marker='o', title='Estimated share of site volume — top 10 categories')
    plt.ylabel('share'); plt.legend(fontsize=7, bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

In [ ]:
# --- CELL 13: EXPORT ---
out = f'{BASE_DIR}/mlc_category_volume_{TODAY}.csv'
feat.sort_values('volume_score', ascending=False).to_csv(out, index=False)
print(f"💾 Saved {out}  |  total API calls this run: {ml.calls}")

### Caveats (read before trusting the numbers)
* MercadoLibre never exposes unit sales for other sellers' items. Everything here is a **proxy** built from MercadoLibre's own best-seller ranking plus demand and supply signals. Treat it as a ranking, not as exact units.
* `parent_share` assumes parent and child rankings use the same sales metric. Check `attribution_coverage`: a median above ~80% means the lists match well.
* `root_weight` is the weakest link when only trends are available. Enable reviews (cell 7) and run for several days so review velocity can replace it.
* Some endpoints can become blocked even with a valid token. Cell 3 tells you what is live each day.